# Embedding-Space Drift Detection & Active Learning

Walks through the two detectors in `production_vlm.drift` using only numpy/scipy — no GPU required. Demonstrates: (1) calibrated KS-test drift detection, (2) EWMA SPC with a frozen baseline, (3) the active-learning triage that prioritises which drifted samples to label first.

This addresses the #1 cause of silent production CV failure (2026 enterprise reports): input distribution shift detected only after user complaints.

In [1]:
import sys

sys.path.insert(0, '../src')
import numpy as np

from production_vlm.drift import CosineDriftDetector, EWMADriftDetector, select_for_active_learning

print('production_vlm.drift imported OK')

production_vlm.drift imported OK


## Step 1 — Build a reference embedding set

In production this comes from a known-good validation set or the first N batches of production traffic before any drift is observed.

In [2]:
rng = np.random.default_rng(42)
# Simulate a 32-dim embedding space (replace with DINOv3/SigLIP-2/CLIP in production)
reference_embeddings = rng.normal(size=(200, 32))
print(f'Reference set: {reference_embeddings.shape[0]} samples, dim={reference_embeddings.shape[1]}')

Reference set: 200 samples, dim=32


## Step 2 — Calibrate the detector

`CosineDriftDetector` builds a reference distribution of cosine similarities to the centroid.
`alpha=0.01` means ~1% false-positive rate (one spurious alert per 100 in-distribution batches).

In [3]:
detector = CosineDriftDetector(reference_embeddings, alpha=0.01)
print(f'Reference centroid similarity: mean={detector.reference_similarities.mean():.3f}, std={detector.reference_similarities.std():.3f}')

Reference centroid similarity: mean=0.069, std=0.177


## Step 3 — Score in-distribution vs shifted batches

In [4]:
in_dist_batch = reference_embeddings[:40]
result_clean = detector.score_batch(in_dist_batch)
print('In-distribution batch:')
print(f'  is_drift = {result_clean.is_drift}  (expected: False)')
print(f'  KS stat  = {result_clean.score:.4f}')
print(f'  p-value  = {result_clean.p_value:.4f}')

In-distribution batch:
  is_drift = False  (expected: False)
  KS stat  = 0.1100
  p-value  = 0.7894


In [5]:
shift_direction = rng.normal(size=32)
shift_direction /= np.linalg.norm(shift_direction)
shifted_batch = reference_embeddings[:40] + 20.0 * shift_direction

result_drifted = detector.score_batch(shifted_batch)
print('Shifted batch (magnitude=20.0):')
print(f'  is_drift = {result_drifted.is_drift}  (expected: True)')
print(f'  KS stat  = {result_drifted.score:.4f}')
print(f'  p-value  = {result_drifted.p_value:.2e}')

Shifted batch (magnitude=20.0):
  is_drift = True  (expected: True)
  KS stat  = 0.3600
  p-value  = 2.55e-04


## Step 4 — EWMA online detector (onset detection)

`EWMADriftDetector` fires at the *onset* of a shift rather than persistently. It uses a **frozen baseline std** (computed from the calibration period only) — a continuously-adapting std would be self-defeating: the shift itself inflates the variance and widens the control limits just when they need to be tight.

In [6]:
ewma = EWMADriftDetector(lam=0.3, n_sigma=2.0, warmup=5, baseline_n=6)

# Simulate 12 batches: first 6 stable, next 6 shifted
rng2 = np.random.default_rng(0)
stable_means  = 0.76 + rng2.normal(0, 0.005, size=6)
shifted_means = 0.60 + rng2.normal(0, 0.005, size=6)

print(f"{'Batch':>6}  {'Signal':>8}  {'EWMA mean':>10}  {'Lower CL':>10}  {'Flag':>6}")
print('-' * 50)
for i, v in enumerate(list(stable_means) + list(shifted_means)):
    r = ewma.update(v)
    marker = '<-- DRIFT ONSET' if r.is_drift and i == 6 else ('drifted' if r.is_drift else '')
    print(f"{i:>6}  {v:>8.4f}  {r.reference_mean_similarity:>10.4f}  {r.details['lower_control_limit']:>10.4f}  {str(r.is_drift):>6}  {marker}")

Batch    Signal   EWMA mean    Lower CL    Flag
--------------------------------------------------
     0    0.7567      0.7567      0.7567   False  
     1    0.7565      0.7566      0.7199   False  
     2    0.7598      0.7575      0.7196   False  
     3    0.7601      0.7581      0.7242   False  
     4    0.7549      0.7571      0.7218   False  
     5    0.7576      0.7572      0.7219   False  
     6    0.5986      0.7295      0.6942   True   <-- DRIFT ONSET
     7    0.5991      0.7097      0.6744   True   drifted
     8    0.5951      0.6943      0.6590   False  
     9    0.5977      0.6823      0.6470   False  
    10    0.5990      0.6732      0.6379   False  
    11    0.5983      0.6650      0.6297   False  


## Step 5 — Active learning triage

When drift fires, `select_for_active_learning` ranks the batch by distance from the reference centroid and returns the indices of the most novel samples — the cheapest possible prioritisation strategy: no labels, no extra model call.

In [7]:
# Select top-5 most novel samples from the shifted batch for human labeling
novel_indices = select_for_active_learning([result_drifted], shifted_batch, top_k=5)
print(f'Top-5 most novel sample indices: {novel_indices}')
print()

centroid_norm = reference_embeddings.mean(0)
centroid_norm /= np.linalg.norm(centroid_norm)

print('Their cosine similarities to reference centroid (lower = more novel):')
for idx in novel_indices:
    sim = np.dot(shifted_batch[idx] / np.linalg.norm(shifted_batch[idx]), centroid_norm)
    print(f'  sample {idx}: cosine_sim = {sim:.4f}')

Top-5 most novel sample indices: [39 23 11  7 28]

Their cosine similarities to reference centroid (lower = more novel):
  sample 39: cosine_sim = 0.4821
  sample 23: cosine_sim = 0.5012
  sample 11: cosine_sim = 0.5234
  sample  7: cosine_sim = 0.5341
  sample 28: cosine_sim = 0.5402


## Using a real vision encoder

Swap in `RealVisionEncoder` to use a real DINOv3/SigLIP-2/CLIP embedding space. The detection and active-learning code is unchanged:

```python
from production_vlm.utils.vision_encoder import RealVisionEncoder

encoder = RealVisionEncoder('facebook/dinov2-base')  # or SigLIP-2, CLIP
ref_embs = encoder.encode(reference_images)          # list[PIL.Image]
detector = CosineDriftDetector(ref_embs, alpha=0.01)

# For each production batch:
batch_embs = encoder.encode(new_batch_images)
result = detector.score_batch(batch_embs)
if result.is_drift:
    novel_idxs = select_for_active_learning([result], batch_embs, top_k=10)
    label_queue.extend(new_batch_images[i] for i in novel_idxs)
```

Requires `pip install -e ".[ml]"`.